In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# AI Agent Multi-Domain Task Automation (Ultra-Light Version)
Minimal, fast-running version for Kaggle. Demonstrates text and image automation using pre-trained models. Runs in <10 seconds.

In [2]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3
!pip install --upgrade transformers torch torchvision


Found existing installation: protobuf 6.33.0
Uninstalling protobuf-6.33.0:
  Successfully uninstalled protobuf-6.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; pyth

In [3]:
import torch
from transformers import pipeline
from torchvision.datasets import CIFAR10

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")


2025-11-25 17:13:33.149258: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764090813.385186      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764090813.450387      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Device: CPU


In [4]:
dataset = CIFAR10(root='./data', download=True, train=False)
images = [dataset[i][0] for i in range(2)]
text_samples = [
    "The quick brown fox jumps over the lazy dog.",
    "I love this product!"
]
print("Data loaded successfully.")


100%|██████████| 170M/170M [00:10<00:00, 16.0MB/s]


Data loaded successfully.


In [5]:
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=device)
sentiment_analyzer = pipeline("sentiment-analysis", device=device)
image_classifier = pipeline("image-classification", model="google/vit-base-patch16-224", device=device)
print("Models loaded successfully.")


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/460M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Device set to use cpu


Models loaded successfully.


In [6]:
class MultiDomainAgent:
    def __init__(self, summarizer, sentiment_analyzer, image_classifier):
        self.summarizer = summarizer
        self.sentiment_analyzer = sentiment_analyzer
        self.image_classifier = image_classifier

    def automate(self, input_data, domain, task):
        try:
            if domain == "text":
                if task == "summarize":
                    return self.summarizer(input_data, max_length=30)[0]['summary_text']
                elif task == "sentiment":
                    return self.sentiment_analyzer(input_data)[0]['label']

            elif domain == "image" and task == "classify":
                return self.image_classifier(input_data)[0]['label']

            else:
                return "Unsupported task or domain"

        except Exception as e:
            return f"Error handled safely: {str(e)}"


agent = MultiDomainAgent(summarizer, sentiment_analyzer, image_classifier)
print("AI Agent initialized.")


AI Agent initialized.


In [7]:
print("=== Testing AI Agent ===")

summary = agent.automate(text_samples[0], "text", "summarize")
sentiment = agent.automate(text_samples[1], "text", "sentiment")
classification = agent.automate(images[0], "image", "classify")

print("Summary:", summary)
print("Sentiment:", sentiment)
print("Image Class:", classification)

print("✅ Multi-domain automation successful")


Your min_length=56 must be inferior than your max_length=30.
Your max_length is set to 30, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)


=== Testing AI Agent ===


/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (30). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


Summary:  The quick brown fox jumps over the lazy dog. The fox is the latest in a series of foxes to jump over the dog .
Sentiment: POSITIVE
Image Class: tabby, tabby cat
✅ Multi-domain automation successful


In [8]:
try:
    print("Notebook ready for export.")
except Exception as e:
    if "GetPrototype" in str(e):
        print("⚠ Kaggle protobuf export bug suppressed.")
    else:
        raise


Notebook ready for export.
